In [3]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/Project/'
netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
true_time=netCDF['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);

#Restricts the timesteps of the data from timesteps0 to 140
data=netCDF.isel(time=np.arange(0,140+1))
parcel=parcel.isel(time=np.arange(0,140+1))

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))

In [8]:
# Loading Important Variables
##############
if 'emptylike' not in globals():
    print('loading neccessary variables')
    variable='w'; w_data=data[variable] #get w data
    w_data=w_data.interp(zf=data['zh']).data #interpolation w data z coordinate from zh to zf
    variable='qv'; qv_data=data[variable].data # get qc data
    variable='qc'; qc_data=data[variable].data # get qc data
    variable='qi'; qi_data=data[variable].data # get qc data
    qc_plus_qi=qc_data+qi_data
    print('done')
    empty_like=True

loading neccessary variables
done


In [5]:
#Eulerian Cloudy Updrafts
##############
w_thresh=0.1
qcqithresh=1e-6
D=np.zeros_like(w_data)
where1=np.where((w_data>=w_thresh)&(qc_plus_qi>=qcqithresh))
where1

(array([  6,   6,   6, ..., 140, 140, 140]),
 array([ 7,  7,  7, ..., 31, 31, 31]),
 array([45, 46, 48, ..., 64, 99, 99]),
 array([260, 260, 260, ..., 442, 420, 421]))

In [6]:
#Lagrangian Position Arrays
##############
def grid_location(x,y,z): #faster
    #finding xf and yf
    ybins=data['yf'].values*1000; dy=ybins[1]-ybins[0] #1000
    xbins=data['xf'].values*1000; dx=xbins[1]-xbins[0] #1000
    dy=np.round(dy);dx=np.round(dx)

    #digitizing
    zf=data['zf'].values*1000; which_zh=np.searchsorted(zf,z)-1; which_zh=np.where(which_zh == -1, 0, which_zh) #finds which z layer parcel in 
    if which_zh.ndim==0:
        which_zh=np.array([which_zh])
    which_yh=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0]
    which_xh=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0]

    #fixing boundaries
    which_zh[np.where(which_zh==len(data['zh']))]-=1
    which_yh[np.where(which_yh==len(data['yh']))]-=1
    which_xh[np.where(which_xh==len(data['xh']))]-=1
    return which_zh,which_yh,which_xh
x=parcel['x'].data;y=parcel['y'].data;z=parcel['z'].data
Z,Y,X=grid_location(x,y,z)

In [ ]:
R=np.zeros_like(Z)

max_count=200
start_time = time.time()    
for count,p in enumerate(np.arange(A.shape[1])):
    condz=Z[where1[0],p]==where1[1]
    condy=Y[where1[0],p]==where1[2]
    condx=X[where1[0],p]==where1[3]
    where2=np.where(condz&condy&condx)

    #find (t,p) to index
    t_inds=where1[0][where2]
    p_ind=p

    #indexing T(t,p)
    A[t_inds,p]=1

    if np.mod(count,1000)==0: print(f'p={p}/125000')
    # if count==max_count: break

end_time = time.time()
print(f"Time taken: {end_time - start_time:.6f} seconds")
secs_per_p=(end_time-start_time)/max_count #seconds per parcel
tot_secs=secs_per_p*len(parcel['xh']) #seconds for 1.25e5 parcels
tot_mins=tot_secs/60**2
tot_mins #19 mins calculated from 566 parcels

In [18]:
#*** Use Job Array and then Sum A Arrays Together Later ***

566

In [9]:
# Saving Data
##############
import h5py
with h5py.File(dir+'lagrangian_binary_threshold.h5', 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('A', data=R) #binary array
    f.create_dataset('Z', data=Z)
    f.create_dataset('Y', data=Y)
    f.create_dataset('X', data=X)

In [10]:
# Reading Back Data Later
##############
import h5py
with h5py.File(dir+'lagrangian_binary_threshold.h5', 'r') as f:
    # Load the dataset by its name
    R = f['R'][:]
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]